# PDF to Chroma Pipeline

This notebook loads a PDF, splits it into chunks, stores the chunks in Chroma, and runs a retrieval query.

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

## 1. Prepare Paths and Configuration

In [ ]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

In [ ]:
env_path = project_root / ".env"
pdf_path = project_root / "documents" / "beyond-chatbots-ai-agents-next-real-shift.pdf"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"
collection_name = "rag-pipeline"

print(f"PDF path: {pdf_path}")
print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

In [ ]:
load_dotenv(dotenv_path=env_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {env_path}")

In [ ]:
# Reuse the same embedding model as the other notebooks in this repo.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("Embedding model is ready.")

## 2. Add Small Display Helpers

In [ ]:
def preview_text(text, limit=120):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print retrieved documents using page metadata and a text preview."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. page={doc.metadata.get('page')} | source={doc.metadata.get('source')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Load the PDF

In [ ]:
loader = PyPDFLoader(str(pdf_path))

In [ ]:
docs = loader.load()
print(f"Total pages loaded: {len(docs)}")

In [ ]:
docs[0].page_content

In [ ]:
print(f"First page preview: {preview_text(docs[0].page_content)}")
print(f"First page metadata: {docs[0].metadata}")

## 4. Split the PDF into Chunks

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
)

In [ ]:
chunked_docs = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunked_docs)}")

In [ ]:
print(f"First chunk preview: {preview_text(chunked_docs[0].page_content)}")
print(f"First chunk metadata: {chunked_docs[0].metadata}")

## 5. Store Chunks in Chroma

In [ ]:
collection_name

In [ ]:
persist_directory

In [ ]:
vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    collection_name=collection_name,
    persist_directory=str(persist_directory),
)

print(f"Stored {len(chunked_docs)} chunks in the '{collection_name}' collection.")

## 6. Retrieve Relevant Chunks

In [ ]:
query = "How do AI agents use tools and memory?"
query

In [ ]:
results = vector_store.similarity_search(query, k=3)

print(f"Query: {query}\n")
print_documents("Retrieved chunks:", results)

In [ ]:
retrieved_docs = vector_store.similarity_search_with_score(query, k=2)

for doc, score in retrieved_docs:
    print(f"Score: {score:.4f}")
    print(f"Content preview: {doc.page_content}")
    print(f"page_no. {doc.metadata.get("page_label")}")
    print()